# 대여 GPU 학습 (Vast.ai / RunPod 등): LoRA → full fine-tuning

Google Drive 없이 일반 Jupyter 환경에서 프로젝트 학습 스크립트를 실행하는 노트북.
같은 번들로 **LoRA(v8 설정)** 와 **UNet full fine-tuning(Plan B)** 을 모두 돌릴 수 있다. `MODE`만 바꾼다.

## 추천 인스턴스
| 용도 | GPU | 비고 |
|---|---|---|
| LoRA | RTX 4090 (24GB) | 현재 LoRA 코드는 fp32 UNet이라 fp32 성능이 높은 4090이 유리 (TF32 켬) |
| full FT | A100 80GB | `train_full.yaml` 기본(배치 8, bf16). 24GB GPU는 `train_full_24gb.yaml`(8-bit Adam) 자동 적용 |

- 이미지(템플릿): PyTorch가 들어 있는 공식 템플릿 (Vast.ai "PyTorch", RunPod "RunPod PyTorch"). Python 3.11 이상.
- 디스크: 40GB 이상 (SD 1.5 가중치 약 4GB + 번들 약 720MB + 체크포인트).
- Vast.ai는 신뢰도 점수가 높은 호스트 또는 datacenter 표시 매물을 고른다.

## 준비 (로컬 PC)
1. 변경 사항 커밋 → `python scripts/pack_for_colab.py` → `outputs/colab_bundle.zip`
2. 인스턴스의 Jupyter 파일 브라우저로 `/workspace/`에 업로드 (또는 `scp -P <포트> outputs/colab_bundle.zip root@<호스트>:/workspace/`)
3. 이 노트북도 같은 방식으로 올려 연 뒤 위에서부터 차례로 실행한다.

## 결과 받기
마지막 셀이 `/workspace/results_<run>.zip`(샘플, 로그, 최신 체크포인트)을 만든다. Jupyter 파일 브라우저에서 내려받거나 scp로 가져온다.
**인스턴스를 삭제하면 디스크도 지워지므로** 결과를 받은 뒤 삭제한다. 정지 상태에서도 디스크 요금이 나온다.

In [ ]:
# 설정
MODE = "lora"          # "lora" | "full"
RUN_OVERFIT = False    # True면 본 학습 전에 32장 overfit 테스트 (full 모드 권장)
BUNDLE_PATH = "/workspace/colab_bundle.zip"
WORK_DIR = "/workspace/sde"
OUTPUT_DIR = "/workspace/outputs"   # 학습 출력 (인스턴스 디스크)
SEED = 42

## 1. GPU / Python 확인과 설정 프리셋 선택

In [ ]:
import json, os, re, subprocess, sys

smi = subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout
print(smi)
driver_cuda = float(re.search(r"CUDA Version:\s*([\d.]+)", smi).group(1))
print(f"Python {sys.version.split()[0]}, 드라이버 지원 CUDA {driver_cuda}")
assert sys.version_info >= (3, 11), "requirements.txt 고정 버전은 Python 3.11 이상이 필요하다 (다른 템플릿 선택)"

## 2. 번들 풀기

In [ ]:
import zipfile

os.makedirs(WORK_DIR, exist_ok=True)
with zipfile.ZipFile(BUNDLE_PATH) as zf:
    zf.extractall(WORK_DIR)
os.chdir(WORK_DIR)
print(json.load(open("BUNDLE_INFO.json")))

## 3. 패키지 설치
torch는 프로젝트 고정 버전(`requirements-torch.txt`)과 다르면 다시 설치한다. 드라이버가 CUDA 13 이상이면 cu130, 아니면 cu126 빌드를 쓴다.

In [ ]:
want = dict(line.strip().split("==") for line in open("requirements-torch.txt") if "==" in line and not line.startswith("#"))
try:
    import torch
    have = torch.__version__.split("+")[0]
except ImportError:
    have = None
if have != want["torch"]:
    index = "cu130" if driver_cuda >= 13.0 else "cu126"
    print(f"torch {have} -> {want['torch']} ({index}) 설치")
    !pip install -q -r requirements-torch.txt --index-url https://download.pytorch.org/whl/{index}
else:
    print(f"torch {have} 그대로 사용")
!pip install -q -r requirements.txt

torch를 새로 설치했다면 **커널을 재시작**하고 설정 셀, 1번, 2번 셀을 다시 실행한 뒤 4번으로 넘어간다.

## 4. 환경 점검 (브라우저 검사 생략)

In [ ]:
import torch

!python3 scripts/check_env.py --skip-browser
props = torch.cuda.get_device_properties(0)
vram_gb = props.total_memory / 1024**3
bf16 = torch.cuda.is_bf16_supported()
print(f"{props.name}, VRAM {vram_gb:.1f}GB, bf16={bf16}")

## 5. 실행할 설정 조합

In [ ]:
with open("configs/rental_paths.yaml", "w", encoding="utf-8") as f:
    f.write(f"paths:\n  output_dir: {OUTPUT_DIR}\n")

if MODE == "lora":
    MODULE = "src.train.train_lora"
    CONFIGS = ["configs/base.yaml", "configs/train_lora.yaml", "configs/train_lora_v8.yaml",
               "configs/train_lora_rental.yaml"]
    RUN_NAME = "lora_short_v8_rental"
    OVERFIT = ["configs/train_lora_overfit.yaml", "configs/train_lora_overfit_vpred.yaml",
               "configs/train_lora_overfit_v8.yaml"]
elif MODE == "full":
    MODULE = "src.train.train_full"
    if not bf16:
        preset = ["configs/train_full_t4.yaml"]
    elif vram_gb < 40:
        preset = ["configs/train_full_24gb.yaml"]
    else:
        preset = []
    CONFIGS = ["configs/base.yaml", "configs/train_lora.yaml", "configs/train_full.yaml", *preset]
    RUN_NAME = "full_short"
    OVERFIT = ["configs/train_full_overfit.yaml"]
else:
    raise ValueError(MODE)
print(MODULE, CONFIGS)

## 6. (선택) overfit 테스트
학습 32장으로 파이프라인을 확인한다. 그리드 아래쪽 4행(학습 샘플)이 정답과 거의 같아지면 통과.

In [ ]:
import glob
from IPython.display import Image, display

def show_run(run_name: str) -> None:
    run_dir = f"{OUTPUT_DIR}/{run_name}"
    for line in open(f"{run_dir}/samples/metrics.jsonl"):
        r = json.loads(line)
        print(f"step {r['step']:>5}: val MSE {r['val_mse']:.4f} / train MSE {r['train_mse']:.4f}")
    latest = sorted(glob.glob(f"{run_dir}/samples/step_*.png"))[-1]
    print(latest)
    display(Image(latest, width=512))

if RUN_OVERFIT:
    overfit_configs = CONFIGS + OVERFIT + ["configs/rental_paths.yaml"]
    !python3 -m {MODULE} --seed {SEED} --config {" ".join(overfit_configs)}
    overfit_run = "full_overfit" if MODE == "full" else "lora_overfit_v8_conv"
    show_run(overfit_run)

## 7. 본 학습
full 모드는 출력 폴더에 체크포인트가 있으면 이어서 학습한다. LoRA는 재개를 지원하지 않는다 (같은 run_name이면 로그가 이어 붙으므로 폴더를 지우거나 이름을 바꾼 뒤 다시 실행).
학습 중 진행 상황은 8번 셀을 별도로 실행해 보거나 터미널에서 `tail -f /workspace/outputs/<run>/log.jsonl` 로 본다.

In [ ]:
main_configs = CONFIGS + ["configs/rental_paths.yaml"]
resume = ""
if MODE == "full":
    ckpts = sorted(glob.glob(f"{OUTPUT_DIR}/{RUN_NAME}/checkpoints/step_*"))
    resume = f"--resume {ckpts[-1]}" if ckpts else ""
print("재개:", resume or "처음부터")
!python3 -m {MODULE} --seed {SEED} --config {" ".join(main_configs)} {resume}

## 8. 학습 경과 보기

In [ ]:
show_run(RUN_NAME)
!tail -n 3 {OUTPUT_DIR}/{RUN_NAME}/log.jsonl

## 9. 추론 (학습에 쓰지 않은 HTML)

In [ ]:
CKPT = sorted(glob.glob(f"{OUTPUT_DIR}/{RUN_NAME}/checkpoints/step_*"))[-1]
FLAG = "--lora" if MODE == "lora" else "--full"
HTML = '<body style="margin:0;padding:16px;background:#def;color:#000"><h1>Hello world</h1><div style="width:50%;height:120px;background:#e33"></div></body>'
!python3 -m src.infer.generate {FLAG} {CKPT} --html-text '{HTML}' --out {OUTPUT_DIR}/infer/{RUN_NAME}.png --seed {SEED}
display(Image(f"{OUTPUT_DIR}/infer/{RUN_NAME}.png", width=512))

## 10. 결과 묶기 (내려받기용)

In [ ]:
import shutil

run_dir = f"{OUTPUT_DIR}/{RUN_NAME}"
pack_dir = f"/workspace/results_{RUN_NAME}"
shutil.rmtree(pack_dir, ignore_errors=True)
shutil.copytree(run_dir, pack_dir, ignore=shutil.ignore_patterns("checkpoints"))
ckpts = sorted(glob.glob(f"{run_dir}/checkpoints/step_*"))
if ckpts:
    shutil.copytree(ckpts[-1], f"{pack_dir}/checkpoints/{os.path.basename(ckpts[-1])}")
archive = shutil.make_archive(pack_dir, "zip", pack_dir)
print(archive, f"{os.path.getsize(archive) / 1024**2:.0f}MB")